# Image Reproduction: F-INR vs Baseline

This notebook runs a representative image encoding experiment on the bundled asset `assets/images/kodim23.png`.

It compares:
- the clean factorized image F-INR,
- the clean baseline coordinate MLP,

using the same public JSON configs that the command-line interface uses.

In [ ]:
import importlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for entry in (ROOT, ROOT / "scripts", ROOT / "src"):
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

import f_inr_img

importlib.reload(f_inr_img)
run_image_experiment = f_inr_img.run_image_experiment

finr_config = json.load((ROOT / "configs/image/kodim23_finr.json").open())
baseline_config = json.load((ROOT / "configs/image/kodim23_baseline.json").open())
asset_path = ROOT / finr_config["img_path"]
print(f_inr_img.__file__)
asset_path

In [ ]:
image = Image.open(asset_path)
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.title(f"Asset: {asset_path.relative_to(ROOT)}")
plt.axis("off")
plt.show()

In [ ]:
finr_summary, _ = run_image_experiment(finr_config)


In [ ]:
baseline_summary, _ = run_image_experiment(baseline_config)
summaries = [finr_summary, baseline_summary]
print(json.dumps(summaries, indent=2))
print(
    "Speedup (baseline / F-INR):",
    round(baseline_summary["elapsed_seconds"] / finr_summary["elapsed_seconds"], 3),
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(Image.open(Path(finr_summary["output_dir"]) / "input.png"))
axes[0].set_title("Input")
axes[1].imshow(Image.open(Path(finr_summary["output_dir"]) / "reconstruction.png"))
axes[1].set_title("F-INR Reconstruction")
axes[2].imshow(Image.open(Path(baseline_summary["output_dir"]) / "reconstruction.png"))
axes[2].set_title("Baseline Reconstruction")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()